In [91]:
import importlib
import model_adapters

# 1) Reload the *submodule* that contains the class
import model_adapters.llavahf_adapter as llavahf_adapter
importlib.reload(llavahf_adapter)

# 2) (Optional but nice) reload the package __init__ so its "from .llavahf_adapter import LlavaHFAdapter" sees the new version
importlib.reload(model_adapters)

# 3) Re-import the class into the notebook namespace
from model_adapters import LlavaHFAdapter

In [96]:
# Cell 1: imports, eval functions, and evaluate()

import os
import json
import yaml
from pathlib import Path

from tqdm import tqdm
import datasets
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

import model_adapters
from utils import DEFAULT_PROMPTS
from utils import (
    eval_web_caption,
    eval_heading_ocr,
    eval_element_ocr,
    eval_action_prediction,
    eval_element_ground,
    eval_action_ground,
    eval_webqa,
)
from utils.constants import *

# Optional: disable flash attention if needed
os.environ["HF_USE_FLASH_ATTENTION"] = "0"

# Task → metric mapping (same as run.py)
eval_metric = {
    CAPTION_TASK: eval_web_caption,
    HEADING_OCR_TASK: eval_heading_ocr,
    WEBQA_TASK: eval_webqa,
    ELEMENT_OCR_TASK: eval_element_ocr,
    ELEMENT_GROUND_TASK: eval_element_ground,
    ACTION_PREDICTION_TASK: eval_action_prediction,
    ACTION_GROUND_TASK: eval_action_ground,
}


def evaluate(
    model_adapter: model_adapters.BaseAdapter,
    prompt: str,
    dataset: datasets.Dataset,
    task_type: str,
    **kwargs,
):
    """Same logic as run.py, but notebook-friendly."""
    preds, golds = [], []
    print("=" * 80)
    print("Prompt: ", prompt)
    data_size = len(dataset)

    cnt = 0

    for idx_ in tqdm(range(data_size), desc=task_type):
        # cnt += 1
        # if cnt == 10:
        #     break
        sample = dataset[idx_]

        if task_type in [CAPTION_TASK, HEADING_OCR_TASK]:
            cur_prompt = prompt
        elif task_type == WEBQA_TASK:
            cur_prompt = prompt.format(question=sample["question"])
        elif task_type == ELEMENT_OCR_TASK:
            cur_prompt = prompt.format(bbox_ratio=sample["bbox"])
        elif task_type == ELEMENT_GROUND_TASK:
            cur_prompt = prompt.format(element_desc=sample["elem_desc"])
        elif task_type == ACTION_PREDICTION_TASK:
            cur_prompt = prompt.format(
                bbox_ratio=sample["bbox"], choices_text=sample["options"]
            )
        elif task_type == ACTION_GROUND_TASK:
            cur_prompt = prompt.format(instruction=sample["instruction"])
        else:
            raise NotImplementedError(f"Task type {task_type} not implemented.")

        # model_adapter is callable: (prompt, image, task_type) -> string
        if type(model_adapter).__name__ == "LlavaHFAdapter" and task_type==WEBQA_TASK:
            # Our custom adapter: give it the full sample
            response = model_adapter(
                cur_prompt,
                sample["image"],
                task_type=task_type,
                question=sample["question"],
            )
        elif type(model_adapter).__name__ == "LlavaHFAdapter" and task_type==ACTION_GROUND_TASK:
            # Our custom adapter: give it the full sample
            response = model_adapter(
                cur_prompt,
                sample["image"],
                task_type=task_type,
                question=sample["instruction"],
            )
        else:
            # All other adapters: original behavior
            response = model_adapter(
                cur_prompt,
                sample["image"],
                task_type=task_type,
            )

        preds.append(response)
        golds.append(sample["answer"])

    scores = eval_metric[task_type](preds, golds)
    return scores, preds, golds


In [3]:
# Cell 2: one-time HF LLaVA init for llava_hf branch

from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

hf_llava_model_id = "llava-hf/llava-v1.6-mistral-7b-hf"  # your chosen HF id

print("Loading HF LLaVA model once...")
hf_llava_processor = LlavaNextProcessor.from_pretrained(hf_llava_model_id)

hf_llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    hf_llava_model_id,
    load_in_8bit=True,   # or False for fp16 if VRAM allows
    device_map="auto",
)
hf_llava_model.config.use_cache = False
hf_llava_model.eval()

print("HF LLaVA model loaded.")

Loading HF LLaVA model once...


Some kwargs in processor config are unused and will not have any effect: image_token, patch_size, vision_feature_select_strategy, num_additional_image_tokens. 
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 4/4 [01:54<00:00, 28.68s/it]

HF LLaVA model loaded.


In [97]:
# Cell 3: experiment settings (replaces argparse + shell env)

# Corresponds to --model_name
model_name = "llava_hf_v16_mistral"  # matches configs/llava_hf_v16_mistral.yaml

# Corresponds to --dataset_name_or_path
dataset_name_or_path = "webbench/WebBench"

# Corresponds to --task_type (comma-separated like in your script)
task_type_str = 'webqa' #"action_ground" #'webqa' #'heading_ocr' #'webqa'#'heading_ocr' # "action_ground"

# Device selection (replaces --gpus 0)
gpu_id = 0
device = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"

# Output root (replaces --output_path output)
output_root = Path("output")
output_root.mkdir(exist_ok=True)

# Expand comma-separated tasks
if "," in task_type_str:
    task_types = [t.strip() for t in task_type_str.split(",")]
else:
    task_types = [task_type_str]

print("Model name:", model_name)
print("Tasks:", task_types)
print("Device:", device)
print("Output root:", output_root)


Model name: llava_hf_v16_mistral
Tasks: ['webqa']
Device: cuda:0
Output root: output


In [98]:
# Cell 4: load model config and build model_adapter (with llava_hf branch)

# Load YAML config for this model
config_path = Path("configs") / f"{model_name}.yaml"
assert config_path.exists(), f"Config not found: {config_path}"

with open(config_path, "r") as f:
    model_config = yaml.load(f, Loader=yaml.FullLoader)

model_path = model_config.get("model_path")
tokenizer_path = model_config.get("tokenizer_path", model_path)
print("model_path:", model_path)
print("tokenizer_path:", tokenizer_path)


def build_model_adapter_from_config(model_config, model_path, tokenizer_path, device):
    """
    Notebook version of run.py's model-loading logic, with a dedicated 'llava_hf' branch.
    """
    model_name_local = model_path.split("/")[-1].lower()

    # --- OpenAI GPT-style models ---
    if "gpt" in model_name_local:
        from openai import OpenAI

        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(client, model_path)

    # --- Gemini ---
    elif "gemini" in model_name_local:
        import google.generativeai as genai

        genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
        model = genai.GenerativeModel(model_path)
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(model)

    # --- Claude ---
    elif "claude" in model_name_local:
        from anthropic import Anthropic

        client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(client, model_path)

    # --- NEW: HF LLaVA branch (must come BEFORE the original 'llava' branch) ---
    elif "llava_hf" in model_name_local:
        # Use the globally initialized hf_llava_model / hf_llava_processor
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(
            hf_llava_model,
            hf_llava_processor,
            use_agent=False,     # turn on VisualCoTAgent for WEBQA
            max_new_tokens=model_config.get("max_new_tokens", 128),
            temperature=model_config.get("temperature", 0.0),
            agent_grid_size=3,
            agent_max_crops=1,
            agent_margin_frac_of_cell=0.2,
            agent_save_dir="./agent_debug_images",   # or None
        )


    # --- Original LLaVA repo branch (unchanged) ---
    elif "llava" in model_name_local:
        from llava.model.builder import load_pretrained_model
        from llava.utils import disable_torch_init
        from llava.mm_utils import get_model_name_from_path

        raw_model_name = get_model_name_from_path(model_path)
        disable_torch_init()
        tokenizer, model, image_processor, context_len = load_pretrained_model(
            model_path,
            None,
            raw_model_name,
            device_map=None,
            device=device,
            load_8bit=True,
        )
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(
            model, tokenizer, context_len, image_processor, model_config["conv_mode"]
        )

    # --- BLIP2 branch ---
    elif "blip2" in model_name_local:
        from transformers import Blip2Processor, Blip2ForConditionalGeneration

        processor = Blip2Processor.from_pretrained(model_path)
        model = Blip2ForConditionalGeneration.from_pretrained(
            model_path,
            device_map=device,
            torch_dtype=torch.float16,
        )
        adapter_cls = getattr(model_adapters, model_config["model_adapter"])
        model_adapter = adapter_cls(model, processor, **model_config)

    # --- Generic HF AutoProcessor / AutoModelForCausalLM branch ---
    else:
        from transformers import AutoConfig, AutoProcessor, AutoTokenizer

        torch_dtype = torch.bfloat16

        try:
            processor = AutoProcessor.from_pretrained(
                tokenizer_path,
                trust_remote_code=True,
            )
            config = AutoConfig.from_pretrained(
                model_path,
                trust_remote_code=True,
                attn_implementation="sdpa",
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                config=config,
                device_map=device,
                torch_dtype=torch_dtype,
                trust_remote_code=True,
            )
            adapter_cls = getattr(model_adapters, model_config["model_adapter"])
            model_adapter = adapter_cls(model, processor, **model_config)
        except Exception as e:
            print("[build_model_adapter] AutoProcessor branch failed, falling back to text-only:", e)
            tokenizer = AutoTokenizer.from_pretrained(
                tokenizer_path,
                trust_remote_code=True,
            )
            config = AutoConfig.from_pretrained(
                model_path,
                trust_remote_code=True,
                attn_implementation="sdpa",
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                config=config,
                device_map=device,
                torch_dtype=torch_dtype,
                trust_remote_code=True,
            )
            adapter_cls = getattr(model_adapters, model_config["model_adapter"])
            model_adapter = adapter_cls(model, tokenizer)

    return model_adapter


# Build the adapter ONCE
model_adapter = build_model_adapter_from_config(
    model_config, model_path, tokenizer_path, device
)
print("Model adapter:", type(model_adapter))


model_path: llava_hf_v16_mistral
tokenizer_path: llava_hf_v16_mistral
Model adapter: <class 'model_adapters.llavahf_adapter.LlavaHFAdapter'>


In [99]:
# Cell 5: run evaluation for each task and save results

# Make model-specific output dir
output_dir = output_root / model_name
output_dir.mkdir(parents=True, exist_ok=True)
print("Output dir:", output_dir)

for task_type in task_types:
    print("\n======================================")
    print(f"Running task: {task_type}")
    print("======================================")

    prompt_key = f"{task_type}_prompt"
    prompt = model_config.get(prompt_key, DEFAULT_PROMPTS[prompt_key])
    print("Using prompt:\n", prompt)

    dataset = datasets.load_dataset(dataset_name_or_path, task_type)["test"]
    print("Dataset size:", len(dataset))

    scores, preds, golds = evaluate(
        model_adapter=model_adapter,
        prompt=prompt,
        dataset=dataset,
        task_type=task_type,
    )

    score_str = ", ".join([f"{k}: {v:.2f}" for k, v in scores.items()])
    print(f"Model: {model_name}, Task: {task_type}, Scores: {score_str}")

    # Save like run.py
    output_res = [
        {"pred": pred, "gold": gold}
        for pred, gold in zip(preds, golds)
    ]
    output_res = [{"score": score_str}] + output_res

    out_path = output_dir / f"{task_type}.json"
    with open(out_path, "w") as f:
        json.dump(output_res, f, indent=2)

    print("Saved results to:", out_path)


Output dir: output/llava_hf_v16_mistral

Running task: webqa
Using prompt:
 {question}
You should directly tell me your answer in the fewest words possible, and do not output any explanation or any other contents.

Dataset size: 314
Prompt:  {question}
You should directly tell me your answer in the fewest words possible, and do not output any explanation or any other contents.



webqa: 100%|██████████| 314/314 [21:10<00:00,  4.04s/it]

Model: llava_hf_v16_mistral, Task: webqa, Scores: f1: 37.36
Saved results to: output/llava_hf_v16_mistral/webqa.json
